# Global 모델링 Stage 4 — 고정 Test 최종 비교

> **경고:** 이 Notebook을 실행하면 고정 Global Test Dataset에 대한 최종 평가가 수행됩니다. Test 결과 확인 이후 Feature, Hyperparameter, Threshold를 변경하지 않습니다.

네 고정 후보(LR/XGB × 1차 42개 Feature/2차 25개 Feature)를 threshold=0.5로 평가한다. 결과를 자동 해석하거나 최종 모델을 선택하지 않는다.

In [ ]:
import importlib, json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

ROOT = Path(os.environ.get('KHUDA_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'code').is_dir():
    if ROOT.parent == ROOT: raise RuntimeError('KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 실행하세요.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'): del sys.modules['code']
for module in ('code.model.final_evaluation', 'code.evaluation.bootstrap', 'code.evaluation.importance'):
    if module in sys.modules: importlib.reload(sys.modules[module])
from code.evaluation.bootstrap import bootstrap_confidence_intervals, paired_bootstrap_f1_difference
from code.evaluation.importance import calculate_feature_importance
from code.model.final_evaluation import (fit_final_candidates, load_final_candidates, load_stage4_cv_f1, predict_final_candidates, save_final_test_artifacts, summarize_final_predictions)
from code.pipeline.saved_results import load_saved_global_train_test
from code.preprocess.build_features import load_feature_config

RESULT_ROOT = ROOT / 'data' / 'result' / 'baseline_42features'
DATASET_PATH = RESULT_ROOT / 'datasets' / 'local_dataset.parquet'
SPLIT_PATH = RESULT_ROOT / 'splits' / 'split_ids.csv'
STAGE_1_DIR = RESULT_ROOT / 'modeling' / 'stage_1'
STAGE_3_DIR = RESULT_ROOT / 'modeling' / 'stage_3_local_healthcare' # 그룹명 수정
STAGE_3_5_DIR = RESULT_ROOT / 'modeling' / 'stage_3_5_local_healthcare' # 그룹명 수정
OUTPUT_DIR = RESULT_ROOT / 'modeling' / 'stage_4_local_healthcare' # 그룹명 수정 
FEATURE_CONFIG = ROOT / 'code' / 'config' / 'features.yaml'
MODEL_CONFIG = ROOT / 'code' / 'config' / 'model_config.yaml'
THRESHOLD, BOOTSTRAP_REPEATS, PERMUTATION_REPEATS = 0.5, 1000, 20
LABELS = {'lr_stage_1':'LR 튜닝 전', 'lr_stage_2':'LR 튜닝 후', 'xgb_stage_1':'XGBoost 튜닝 전', 'xgb_stage_2':'XGBoost 튜닝 후'}
COLORS = {'lr_stage_1':'#7a7a7a', 'lr_stage_2':'#0066cc', 'xgb_stage_1':'#333333', 'xgb_stage_2':'#2997ff'}
plt.rcParams.update({'figure.facecolor':'#f5f5f7','axes.facecolor':'#ffffff','axes.edgecolor':'#e0e0e0','text.color':'#1d1d1f', 'font.family':'Malgun Gothic'})


In [ ]:
# 이 셀부터 Test를 처음 사용한다.
from code.model.final_evaluation import FinalCandidate
from code.pipeline.run_pipeline import _subset_bundle

# 파라미터 및 피처 불러오기
with (STAGE_3_DIR / "best_params.json").open(encoding="utf-8") as file:
    stage_3_params = json.load(file)
with (STAGE_3_5_DIR / "final_refined_params.json").open(encoding="utf-8") as file:
    refined_params = json.load(file)
selected = pd.read_csv(STAGE_3_DIR / 'selected_features.csv')["feature"].tolist()

# 내부적으로는 stage_1, stage_2 키를 유지해서 뒤쪽 하드코딩된 시각화/저장 코드가 고장나지 않게 함
candidates = [
    FinalCandidate("lr_stage_1", "logistic_regression", "stage_1", tuple(selected), stage_3_params["logistic_regression"]),
    FinalCandidate("xgb_stage_1", "xgboost", "stage_1", tuple(selected), stage_3_params["xgboost"]),
    FinalCandidate("lr_stage_2", "logistic_regression", "stage_2", tuple(selected), refined_params["logistic_regression"]),
    FinalCandidate("xgb_stage_2", "xgboost", "stage_2", tuple(selected), refined_params["xgboost"]),
]

# 데이터 불러오고 로컬 직군으로 필터링
train_bundle, test_bundle = load_saved_global_train_test(DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG)
train_bundle = _subset_bundle(train_bundle, train_bundle.metadata['job_group'] == '보건·의료') # 여기 바꾸기
test_bundle = _subset_bundle(test_bundle, test_bundle.metadata['job_group'] == '보건·의료') # 여기 바꾸기

audit = pd.DataFrame([{'train_rows':len(train_bundle.X), 'test_rows':len(test_bundle.X), 'threshold':THRESHOLD, 'bootstrap_repeats':BOOTSTRAP_REPEATS, 'permutation_repeats':PERMUTATION_REPEATS}])
display(audit)
display(pd.DataFrame([{'candidate':c.key, 'model':c.model, 'stage':c.stage, 'n_features':len(c.feature_names), 'best_params':c.params} for c in candidates]))


In [ ]:
# Train 전체에서만 전처리 fit 및 고정 파라미터 학습.
fitted_models = fit_final_candidates(train_bundle, candidates, feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG)

In [ ]:
# 고정 Test 확률 예측과 threshold=0.5 분류.
test_predictions = predict_final_candidates(test_bundle, candidates, fitted_models, threshold=THRESHOLD)

In [ ]:
# Test metric과 CV-vs-Test 비교.
test_summary, confusion_matrices = summarize_final_predictions(test_predictions, candidates, threshold=THRESHOLD)

# stage_1은 버리고 stage_3과 stage_3_5의 CV 점수만 직접 가져오기 (내부 키인 stage_1, stage_2로 맵핑)
s3 = pd.read_csv(STAGE_3_DIR / "healthcare_second_stage_summary.csv") # 여기 바꾸기
s3 = s3.loc[:, ["model", "cv_f1_mean"]].assign(stage="stage_1")
s35 = pd.read_csv(STAGE_3_5_DIR / "final_tuning_summary.csv")
s35 = s35.query("stage == 'stage_3_5'").loc[:, ["model", "cv_f1_mean"]].assign(stage="stage_2")
cv_f1 = pd.concat([s3, s35], ignore_index=True)

final_comparison = test_summary.merge(cv_f1, on=['model','stage'], how='left').rename(columns={'cv_f1_mean':'cv_f1'})
final_comparison['test_f1_minus_cv_f1'] = final_comparison['test_f1'] - final_comparison['cv_f1']
display(final_comparison)


In [ ]:
# SAMPID 단위 bootstrap 1,000회. 같은 사람의 Person-Period는 함께 복원추출된다.
bootstrap_ci = pd.concat([bootstrap_confidence_intervals(frame.y_true, frame.y_proba, frame.SAMPID, threshold=THRESHOLD, n_repeats=BOOTSTRAP_REPEATS, random_state=42).query("metric == 'f1'").assign(candidate=key) for key, frame in test_predictions.items()], ignore_index=True)
pairs = [('lr_stage_2','lr_stage_1'), ('xgb_stage_2','xgb_stage_1'), ('xgb_stage_2','lr_stage_2')]
pairwise_bootstrap = pd.concat([paired_bootstrap_f1_difference(test_predictions[a].y_true, test_predictions[a].y_proba, test_predictions[b].y_proba, test_predictions[a].SAMPID, threshold=THRESHOLD, n_repeats=BOOTSTRAP_REPEATS, random_state=42, comparison=f'{a} - {b}') for a, b in pairs], ignore_index=True)
display(bootstrap_ci); display(pairwise_bootstrap)

In [ ]:
# 2차 25개 Feature held-out Test PI. 재선택에 사용하지 않는 최종 진단 자료다.
candidate_by_key = {candidate.key:candidate for candidate in candidates}
permutation_importance = {}
for key in ('lr_stage_2','xgb_stage_2'):
    candidate = candidate_by_key[key]
    X_test = test_bundle.X.loc[:, list(candidate.feature_names)]
    permutation_importance[key] = calculate_feature_importance(fitted_models[key], X_test, test_bundle.y, scoring='f1', n_repeats=PERMUTATION_REPEATS, random_state=42)
display(permutation_importance['lr_stage_2']); display(permutation_importance['xgb_stage_2'])

In [ ]:
# 사람이 실행했을 때만 Test 결과 artifact를 저장한다.
artifact_paths = save_final_test_artifacts(output_dir=OUTPUT_DIR, summary=final_comparison, bootstrap_ci=bootstrap_ci, pairwise_bootstrap=pairwise_bootstrap, predictions=test_predictions, permutation_importance=permutation_importance, confusion_matrices=confusion_matrices)
display(pd.DataFrame({'artifact':list(artifact_paths), 'path':[str(path) for path in artifact_paths.values()]}))

In [ ]:
# Test F1 및 Accuracy/Precision/Recall/F1.
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].bar([LABELS[k] for k in final_comparison.candidate], final_comparison.test_f1, color=[COLORS[k] for k in final_comparison.candidate]); axes[0].set(title='Global Stage 4: Test F1', xlabel='Candidate', ylabel='Test F1', ylim=(0,1))
metrics = ['test_accuracy','test_precision','test_recall','test_f1']; x=np.arange(4); width=.2
for i,row in enumerate(final_comparison.itertuples()): axes[1].bar(x+(i-1.5)*width,[getattr(row,m) for m in metrics],width,label=LABELS[row.candidate],color=COLORS[row.candidate])
axes[1].set(title='Global Stage 4: Test Metrics', xlabel='Metric', ylabel='Score', ylim=(0,1)); axes[1].set_xticks(x,['Accuracy','Precision','Recall','F1']); axes[1].legend(); plt.tight_layout(); plt.show()

# 네 confusion matrix, 두 모델군별 ROC, 네 모델 PR curve.
fig, axes = plt.subplots(2,2,figsize=(9,8))
for ax,(key,frame) in zip(axes.ravel(),test_predictions.items()): ConfusionMatrixDisplay.from_predictions(frame.y_true,frame.y_pred,ax=ax,colorbar=False); ax.set_title(LABELS[key])
fig.suptitle('Global Stage 4: Test Confusion Matrices'); plt.tight_layout(); plt.show()
for family,keys in [('LR',('lr_stage_1','lr_stage_2')),('XGBoost',('xgb_stage_1','xgb_stage_2'))]:
    fig,ax=plt.subplots(figsize=(6,5))
    for key in keys: RocCurveDisplay.from_predictions(test_predictions[key].y_true,test_predictions[key].y_proba,name=LABELS[key],ax=ax)
    ax.set_title(f'Global Stage 4: {family} ROC Curve'); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(6,5))
for key,frame in test_predictions.items(): PrecisionRecallDisplay.from_predictions(frame.y_true,frame.y_proba,name=LABELS[key],ax=ax)
ax.set_title('Global Stage 4: Test Precision-Recall Curve'); ax.legend(); plt.show()

In [ ]:
# Bootstrap CI, Feature selection 전후 ΔF1, 2차 PI.
ci = bootstrap_ci.set_index('candidate').loc[final_comparison.candidate]
fig,ax=plt.subplots(figsize=(8,4)); values=final_comparison.test_f1.to_numpy(); errors=np.vstack([values-ci.ci95_lower.to_numpy(),ci.ci95_upper.to_numpy()-values]); ax.bar([LABELS[k] for k in final_comparison.candidate],values,yerr=errors,capsize=5,color=[COLORS[k] for k in final_comparison.candidate]); ax.set(title='Global Stage 4: Test F1 with SAMPID Bootstrap 95% CI',xlabel='Candidate',ylabel='Test F1',ylim=(0,1)); plt.tight_layout(); plt.show()
delta = pairwise_bootstrap[pairwise_bootstrap.comparison.isin(['lr_stage_2 - lr_stage_1','xgb_stage_2 - xgb_stage_1'])]; display(delta)
for key in ('lr_stage_2','xgb_stage_2'):
    frame=permutation_importance[key].sort_values('importance_mean'); fig,ax=plt.subplots(figsize=(7,7)); ax.barh(frame.feature,frame.importance_mean,xerr=frame.importance_std,color=COLORS[key]); ax.set(title=f'{LABELS[key]}: Held-out Test Permutation Importance',xlabel='F1 importance',ylabel='Original feature'); plt.tight_layout(); plt.show()

# 발표용 최종 표: 수치만 표시하며 자동 선택·해석을 하지 않는다.
presentation_columns=['model','stage','feature_count','cv_f1','test_accuracy','test_precision','test_recall','test_f1','test_roc_auc','test_average_precision']
presentation = final_comparison.merge(ci[['ci95_lower','ci95_upper']],left_on='candidate',right_index=True,how='left').loc[:,presentation_columns+['ci95_lower','ci95_upper']]
display(presentation)